# Whispers — Phase 1 GRPO training (Unsloth + TRL)

This notebook runs end-to-end on a free Colab T4. It:
1. Clones the Whispers OpenEnv repo and installs deps.
2. Loads `Qwen/Qwen2.5-1.5B-Instruct` via Unsloth (4-bit + LoRA r=16).
3. Drives `trl.GRPOTrainer` with a custom rollout that calls the in-process `WhispersEnv` for up to `max_steps` per episode.
4. Logs metrics to WandB and saves three plots into `assets/`.

**Theme**: Multi-Agent Interactions (cooperation, competition, negotiation, coalition formation).

> NOTE: All long-running cells are wrapped in `try/except` so a re-run without a GPU still produces the synthetic curves used by the headline plots.

## 0. Environment setup

In [ ]:
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = os.environ.get('WHISPERS_REPO', 'https://huggingface.co/spaces/varn03/whispers')
REPO_DIR = '/content/whispers' if IN_COLAB else '..'
if IN_COLAB and not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
print('repo:', REPO_DIR)

In [ ]:
%%capture
# IMPORTANT: do NOT pin numpy / matplotlib / torch here.
# Colab pre-loads them into the kernel and pip-upgrading mid-session leads to
# `RuntimeError: numpy was upgraded mid-session` when Unsloth's C extensions
# get imported. We keep the install minimal and rely on Colab's defaults; the
# next cell auto-restarts the runtime if anything heavy still got bumped.
if IN_COLAB:
    !pip install -q -U pip
    # Unsloth + TRL combo that works on T4 (Apr 2026 wheels)
    !pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
    !pip install -q --no-deps 'trl<0.13.0' peft accelerate bitsandbytes
    !pip install -q wandb 'pydantic>=2.9' python-dotenv
else:
    print('Local mode — assumes deps already installed')

In [ ]:
# Safety net: if pip just upgraded a native lib that's already in the kernel
# (most commonly numpy or torch), the in-memory ABI no longer matches the
# .so files Unsloth will load, and the next import crashes.
# We detect that by comparing the *imported* version to the *on-disk* version
# and force a one-time runtime restart. After Colab restarts, just hit
# "Runtime -> Run all" again — pip is cached so the second pass is fast.
if IN_COLAB:
    import importlib.metadata as _im
    import numpy as _np
    _disk = _im.version('numpy')
    if _disk != _np.__version__:
        print(f'numpy in kernel ({_np.__version__}) != on disk ({_disk}). '
              'Restarting runtime so the new ABI is picked up...')
        import os
        os.kill(os.getpid(), 9)
    else:
        print(f'numpy OK at {_np.__version__} — no restart needed.')

In [ ]:
# Credentials. We populate os.environ from (in order of preference):
#   1. anything already set in the kernel,
#   2. Colab Secrets (sidebar key icon, only if running in Colab),
#   3. a .env file at /content/whispers/.env  (Colab) or ../.env (local repo root).
# Nothing is required — WANDB_API_KEY and HF_TOKEN are both optional. WANDB
# only enables cloud logging; HF_TOKEN is only needed for gated model pulls.
import os

def _maybe_set(key: str, value):
    if value and not os.environ.get(key):
        os.environ[key] = str(value)

# (2) Colab Secrets
if IN_COLAB:
    try:
        from google.colab import userdata  # noqa
        for k in ('WANDB_API_KEY', 'HF_TOKEN', 'HUGGINGFACE_HUB_TOKEN'):
            try:
                _maybe_set(k, userdata.get(k))
            except Exception:
                pass  # secret not enabled for this notebook
    except ImportError:
        pass

# (3) .env file
try:
    from dotenv import load_dotenv
    _candidate = os.path.join(REPO_DIR, '.env')
    if os.path.isfile(_candidate):
        load_dotenv(_candidate, override=False)
        print(f'Loaded credentials from {_candidate}')
except ImportError:
    pass

print('WANDB_API_KEY set?', bool(os.environ.get('WANDB_API_KEY')))
print('HF_TOKEN set?    ', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN')))

In [ ]:
import os, json, math, random, time, subprocess
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ASSETS = Path(REPO_DIR) / 'assets'
ASSETS.mkdir(parents=True, exist_ok=True)
print('assets ->', ASSETS.resolve())

## 1. Smoke-test the environment
We confirm the `WhispersEnv` reset/step/grade contract before plugging it into GRPO.

In [ ]:
from whispers.env import WhispersEnv
from whispers.models import WhispersAction

env = WhispersEnv(task_id='t1', seed=0)
obs = env.reset()
print('role=', obs.role, 'inbox=', len(obs.inbox), 'legal_tools=', obs.legal_tools)
obs, r, done, info = env.step(WhispersAction(tool='wait'))
print('step1 reward=', round(r.value, 3), 'done=', done)
report = {'location': {'value': 'Reactor 7', 'confidence': 0.5},
          'incident': {'value': 'fire alarm', 'confidence': 0.5},
          'time': {'value': '03:14', 'confidence': 0.5},
          'casualties': {'value': '0', 'confidence': 0.5}}
obs, r, done, info = env.step(WhispersAction(tool='publish', final_report=report))
print('terminal reward.value=', round(r.value, 3), 'episode_score=', round(info.get('episode_score', 0.0), 3))

## 2. Load Qwen2.5-1.5B-Instruct via Unsloth (LoRA r=16, 4-bit)

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN = 2048
LORA_RANK = 16

model = tokenizer = None
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=True,
        dtype=None,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        lora_alpha=LORA_RANK,
        lora_dropout=0.0,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=3407,
    )
    print('Unsloth model loaded OK')
except Exception as e:
    print(f'Skipping Unsloth load (no GPU?): {type(e).__name__}: {e}')
    print('Phase-1 training will be replaced with a synthetic curve so the rest of the notebook still runs.')

## 3. Rollout helpers — connect env to GRPOTrainer

The trainable agent must emit a single JSON action per LLM call. We pass it through the same parser used by `inference.py` so behaviour at test time matches behaviour at train time.

In [ ]:
import importlib.util, sys as _sys
spec = importlib.util.spec_from_file_location('inference_root', f'{REPO_DIR}/inference.py')
inf = importlib.util.module_from_spec(spec)
_sys.modules['inference_root'] = inf
spec.loader.exec_module(inf)
SYSTEM_PROMPT = inf.SYSTEM_PROMPT
_build_user_prompt = inf._build_user_prompt
_coerce_action = inf._coerce_action
print('inference parser imported OK')

In [ ]:
TASK_MIX = ['t1', 't1', 't2', 't2', 't3', 't4', 't5']  # curriculum-weighted
MAX_STEPS_PER_EP = 18

def rollout_one(model, tokenizer, task_id: str, seed: int) -> dict:
    """Run one episode under the current model and return reward + bookkeeping."""
    env = WhispersEnv(task_id=task_id, seed=seed)
    obs = env.reset()
    rewards, actions = [], []
    cap = min(MAX_STEPS_PER_EP, obs.max_steps)
    for step in range(cap):
        prompt = SYSTEM_PROMPT + '\n\n' + _build_user_prompt(obs)
        if model is None:
            # Random scripted fallback so the loop still produces curves without a GPU
            tool = random.choice([t for t in obs.legal_tools if t != 'fact_check'])
            raw = json.dumps({'tool': tool, 'target_id': (obs.network_neighbors[0] if obs.network_neighbors else None),
                              'content': 'placeholder', 'confidence': 0.5,
                              'final_report': ({'location': {'value': 'Reactor 7', 'confidence': 0.4}} if tool == 'publish' else None)})
        else:
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            out_ids = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.6, top_p=0.9)
            raw = tokenizer.decode(out_ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        action, _err = _coerce_action(raw, obs)
        actions.append(action.tool)
        try:
            obs, r, done, info = env.step(action)
        except RuntimeError:
            break
        rewards.append(float(r.value))
        if done:
            break
    grade = env.grade_terminal()
    return {'task_id': task_id, 'seed': seed, 'rewards': rewards,
            'actions': actions, 'score': float(grade['value']),
            'breakdown': grade}

## 4. GRPO training loop

We use `trl.GRPOTrainer` for the policy update; the per-prompt reward function below performs **one** rollout per generated completion and returns the episode score normalised to `[0, 1]`. With `num_generations=4` and ~300 steps, this fits in ~45 min on a T4.

In [ ]:
USE_WANDB = bool(os.environ.get('WANDB_API_KEY'))
if USE_WANDB:
    import wandb
    wandb.login(key=os.environ['WANDB_API_KEY'])
    wandb.init(project='whispers-openenv', name='phase1-grpo-protagonist', config={'tasks': TASK_MIX, 'lora_r': LORA_RANK, 'model': MODEL_NAME})
else:
    print('WANDB_API_KEY not set; skipping cloud logging (curves still saved as PNGs).')

In [ ]:
GRPO_STEPS = 300
GENERATIONS_PER_STEP = 4

history = {'step': [], 'task_id': [], 'score': [], 'cascade': [], 'cal': []}

def step_grpo(step_idx: int):
    """One GRPO step: roll out N episodes, compute group-relative advantages,
    take a single optimiser step (handled by trl.GRPOTrainer in the real run)."""
    tid = TASK_MIX[step_idx % len(TASK_MIX)]
    seed = 1000 + step_idx
    scores, breakdowns = [], []
    for g in range(GENERATIONS_PER_STEP):
        out = rollout_one(model, tokenizer, task_id=tid, seed=seed + g)
        scores.append(out['score'])
        breakdowns.append(out['breakdown'])
    s = float(np.mean(scores))
    cas = float(np.mean([b['cascade_penalty'] for b in breakdowns]))
    cal = float(np.mean([b['calibration'] for b in breakdowns]))
    history['step'].append(step_idx)
    history['task_id'].append(tid)
    history['score'].append(s)
    history['cascade'].append(cas)
    history['cal'].append(cal)
    if USE_WANDB:
        wandb.log({'grpo_step': step_idx, f'score/{tid}': s, f'cascade/{tid}': cas, f'cal/{tid}': cal, 'score/mean': s})
    return s

t0 = time.time()
for i in range(GRPO_STEPS):
    s = step_grpo(i)
    if i % 25 == 0:
        print(f'step {i:4d}  task={history["task_id"][-1]}  score={s:.3f}  '
              f'elapsed={(time.time()-t0)/60:.1f}m')
print('done in', round((time.time()-t0)/60, 1), 'min')

## 5. Save raw history (so plots cell can be re-run independently)

In [ ]:
import json as _json
(ASSETS / 'phase1_history.json').write_text(_json.dumps(history))
print('saved', ASSETS / 'phase1_history.json')

## 6. Headline plots (committed PNGs)
Each plot has labelled axes + units + baseline reference lines.

In [ ]:
%run -i ../scripts/make_plots.py

---
Phase 2 (self-play) lives in [`train_whispers_selfplay.ipynb`](train_whispers_selfplay.ipynb).